# Hybrid Intrusion Detection System
## Unsupervised + Supervised Machine Learning on the CICIDS2017 Dataset

**Course:** Machine Learning  
**Dataset:** CICIDS2017 Cleaned (`cicids2017_cleaned.csv`)  
**Methods:** K-Means++, DBSCAN, Hierarchical Clustering, KNN, Decision Tree, Random Forest

---

## Project Overview

Network **Intrusion Detection Systems (IDS)** are a critical component of modern cybersecurity infrastructure. They monitor network traffic and flag suspicious activity that may indicate an attack.

This notebook implements a **Hybrid IDS** by combining:
- **Unsupervised Learning (Phase 2)**: Automatically discover hidden patterns in network traffic *without* using any labels
- **Supervised Learning (Phase 3)**: Train classifiers that can accurately label traffic as *Normal* or a specific *Attack Type*

The key innovation of a hybrid approach is that clustering (unsupervised) adds a **Traffic_Cluster** feature that helps supervised classifiers better distinguish attack types.

## Workflow Summary

| Phase | Description |
|-------|-------------|
| **0** | Dataset Loading & Exploration |
| **1** | Data Preprocessing |
| **2** | Unsupervised Learning & Threat Profiling |
| **3** | Supervised Classification (Linear Regression, KNN, Decision Tree, Random Forest) |
| **4** | Hyperparameter Tuning with GridSearchCV |
| **5** | Model Evaluation |
| **6** | Final Comparison |
| **7** | Conclusion |

---
## Phase 0 — Setup: Import Libraries

Before we begin, we import all the Python libraries needed throughout this project.  
Each library serves a specific purpose:

- **pandas / numpy**: Data manipulation and numerical computing
- **matplotlib / seaborn**: Data visualization and plotting
- **sklearn**: All machine learning algorithms, metrics, and utilities
- **scipy**: Hierarchical clustering dendrogram

In [ ]:
# ============================================================
# IMPORTS — All required libraries for this project
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# --- Preprocessing ---
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA

# --- Unsupervised Learning (Clustering) ---
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

# --- Supervised Learning ---
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier

# --- Evaluation Metrics ---
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report, confusion_matrix)

# --- Hierarchical Clustering Dendrogram ---
from scipy.cluster.hierarchy import dendrogram, linkage

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Set consistent plot style
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 11

print("All libraries imported successfully!")
print("Ready to build a Hybrid Intrusion Detection System!")

---
## Phase 0 — Dataset Loading

### About the CICIDS2017 Dataset

The **CICIDS2017** (Canadian Institute for Cybersecurity Intrusion Detection System 2017) dataset is one of the most widely used benchmarks in network security research.

It contains realistic network traffic captures with **both normal and attack traffic**, including:
- **DoS / DDoS attacks**
- **Port Scanning**
- **Brute Force (SSH, FTP)**
- **Web Attacks (SQL Injection, XSS)**
- **Infiltration attacks**

Each row represents a **network flow** — a summary of all packets between two endpoints over a time window. Features describe statistical properties of the flow (packet sizes, inter-arrival times, byte counts, flags, etc.).

### Loading the CSV

In [ ]:
# ============================================================
# LOAD DATASET
# ============================================================

# Load the cleaned CICIDS2017 CSV file
df = pd.read_csv("cicids2017_cleaned.csv")

print("Dataset loaded successfully!")
print("=" * 50)
print(f"Total rows    : {df.shape[0]:,}")
print(f"Total columns : {df.shape[1]}")
print("=" * 50)

### Preview: First Rows of the Dataset

In [ ]:
# Show the first 5 rows
print("First 5 rows of the dataset:")
df.head()

### Dataset Shape and Data Types

In [ ]:
# Show shape and data types of each column
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print()
print("Data types per column:")
print(df.dtypes)

### Missing Values Check

In [ ]:
# Check for missing values in each column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
})

print("Missing values summary:")
print(missing_df[missing_df["Missing Count"] > 0])
print()
print(f"Total missing values in dataset: {missing.sum()}")

### Auto-Detection of Label Column and Feature Selection

Before we can train models, we need to:
1. **Identify the target (label) column** — the column that tells us if traffic is normal or an attack
2. **Keep only useful numeric features** — remove ID columns, constant columns, and non-informative columns

#### Why remove certain columns?
- **ID columns** (like flow ID or timestamp): These uniquely identify each row and carry no predictive value
- **Constant columns**: If a column has the same value in every row, it provides zero information to the model
- **Columns with >50% missing values**: Too much missing data makes a feature unreliable
- **Non-numeric columns** (except the label): String columns like IP addresses are not directly useful for ML algorithms

In [ ]:
# ============================================================
# AUTO-DETECT LABEL COLUMN
# ============================================================

# Try common label column names in order of preference
possible_labels = ["Label", "Attack Type", "label", "attack_type",
                   "class", "Class", "Category", "category", "attack", "Attack"]

label_col = None
for col_name in possible_labels:
    if col_name in df.columns:
        label_col = col_name
        print(f"Found label column: '{label_col}'")
        break

# If none found, pick the last non-numeric (object dtype) column
if label_col is None:
    object_cols = df.select_dtypes(include=["object"]).columns.tolist()
    if object_cols:
        label_col = object_cols[-1]
        print(f"Auto-detected label column (last string column): '{label_col}'")
    else:
        raise ValueError("Could not automatically find a label column. Please set it manually.")

# Show label distribution
print()
print("Label distribution:")
print(df[label_col].value_counts())
print()
print(f"Number of unique labels: {df[label_col].nunique()}")

In [ ]:
# ============================================================
# IDENTIFY AND REMOVE NON-INFORMATIVE COLUMNS
# ============================================================

# Start with all columns
cols_to_remove = []

# 1. Remove the label column (we store it separately)
# 2. Remove non-numeric columns (e.g., IPs, timestamps as strings)
non_numeric_cols = df.drop(columns=[label_col]).select_dtypes(include=["object"]).columns.tolist()
if non_numeric_cols:
    cols_to_remove += non_numeric_cols
    print(f"Removing non-numeric columns: {non_numeric_cols}")

# 3. Remove columns with more than 50% missing values
missing_ratio = df.isnull().mean()
high_missing_cols = missing_ratio[missing_ratio > 0.5].index.tolist()
if label_col in high_missing_cols:
    high_missing_cols.remove(label_col)
cols_to_remove += high_missing_cols
if high_missing_cols:
    print(f"Removing columns with >50% missing: {high_missing_cols}")

# 4. Remove constant columns (same value in every row)
numeric_df = df.drop(columns=[label_col]).select_dtypes(include=[np.number])
constant_cols = [c for c in numeric_df.columns if numeric_df[c].nunique() <= 1]
cols_to_remove += constant_cols
if constant_cols:
    print(f"Removing constant columns: {constant_cols[:5]} ...")

# Remove all identified columns (unique list)
cols_to_remove = list(set(cols_to_remove))
df_clean = df.drop(columns=cols_to_remove, errors="ignore")

print()
print(f"Columns removed: {len(cols_to_remove)}")
print(f"Dataset shape after column removal: {df_clean.shape}")
print()
print(f"Remaining columns ({df_clean.shape[1]}):")
print(list(df_clean.columns))

In [ ]:
# ============================================================
# EXTRACT FEATURES (X) AND TARGET (y)
# ============================================================

# X = all numeric feature columns (everything except the label)
X_raw = df_clean.drop(columns=[label_col])

# Keep only numeric columns in X
X_raw = X_raw.select_dtypes(include=[np.number])

# y = the attack label column
y_raw = df_clean[label_col].copy()

print(f"Feature matrix X shape : {X_raw.shape}")
print(f"Label vector y shape   : {y_raw.shape}")
print()
print("Feature columns:")
print(list(X_raw.columns))
print()
print("Label column:", label_col)
print("Label sample:", y_raw.unique()[:5])

---
## Phase 1 — Data Preprocessing

Before training any machine learning model, we need to clean and prepare the data carefully.  
Poor preprocessing leads to poor model performance — this is often called *"garbage in, garbage out."*

### Steps in this phase:
1. Handle **missing values** (NaN)
2. Handle **infinite values** (±Inf)
3. Remove **duplicate rows**
4. **Encode** the label column into numbers
5. **Scale** features using StandardScaler

---

### Step 1 — Handle Missing Values

Missing values (NaN) can crash many ML algorithms. We will:
- First replace infinite values (Inf, -Inf) with NaN, since they are equally problematic
- Then drop rows that still contain NaN (since CICIDS2017 is large, we can afford to drop a few rows)

In [ ]:
# ============================================================
# STEP 1 & 2: Handle Infinite and Missing Values
# ============================================================

print("Before cleaning:")
print(f"  Rows          : {X_raw.shape[0]:,}")
print(f"  Missing values: {X_raw.isnull().sum().sum()}")
print(f"  Inf values    : {np.isinf(X_raw).sum().sum()}")

# Replace infinite values with NaN
X_raw.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop rows with any remaining NaN values
# We keep track of which indices to drop
nan_rows = X_raw[X_raw.isnull().any(axis=1)].index
X_raw.drop(index=nan_rows, inplace=True)
y_raw.drop(index=nan_rows, inplace=True)

# Reset index so they align perfectly
X_raw.reset_index(drop=True, inplace=True)
y_raw.reset_index(drop=True, inplace=True)

print()
print("After cleaning:")
print(f"  Rows          : {X_raw.shape[0]:,}")
print(f"  Missing values: {X_raw.isnull().sum().sum()}")
print(f"  Inf values    : {np.isinf(X_raw).sum().sum()}")
print()
print("Data is clean — no missing or infinite values!")

### Step 3 — Remove Duplicate Rows

Duplicate rows can introduce **bias** into training — if the same traffic flow appears multiple times, the model might overfit to it.

In [ ]:
# ============================================================
# STEP 3: Remove Duplicate Rows
# ============================================================

print(f"Rows before removing duplicates: {X_raw.shape[0]:,}")

# Find duplicate rows
dup_mask = X_raw.duplicated()
dup_indices = X_raw[dup_mask].index

# Drop duplicates from both X and y
X_raw.drop(index=dup_indices, inplace=True)
y_raw.drop(index=dup_indices, inplace=True)

# Reset index
X_raw.reset_index(drop=True, inplace=True)
y_raw.reset_index(drop=True, inplace=True)

print(f"Rows after  removing duplicates: {X_raw.shape[0]:,}")
print(f"Duplicates removed: {len(dup_indices):,}")

### Step 4 — Encode the Label Column

Machine learning algorithms require **numerical inputs**. The label column contains strings like `"Normal Traffic"` or `"DoS Hulk"`.

We use `LabelEncoder` from scikit-learn to convert these string labels into integer class codes:
- `"Normal Traffic"` → `0`
- `"DoS Hulk"` → `1`
- etc.

In [ ]:
# ============================================================
# STEP 4: Encode Labels (string → integer)
# ============================================================

# Create a LabelEncoder
le = LabelEncoder()

# Fit and transform the label column
y_encoded = le.fit_transform(y_raw)

print("Label encoding complete!")
print()
print("Class mapping (label → integer code):")
for i, class_name in enumerate(le.classes_):
    count = np.sum(y_encoded == i)
    pct = count / len(y_encoded) * 100
    print(f"  [{i:2d}] {class_name:<30} → {count:>8,} rows ({pct:.1f}%)")
print()
print(f"Total classes: {len(le.classes_)}")

### Step 5 — Feature Scaling with StandardScaler

**Why is scaling important?**

Many ML algorithms are sensitive to the magnitude of feature values:

| Algorithm | Why scaling matters |
|-----------|---------------------|
| **KNN** | Uses Euclidean distance — large-scale features dominate the distance calculation |
| **K-Means** | Also distance-based — features with large ranges skew cluster shapes |
| **DBSCAN** | Distance thresholds (epsilon) depend on a consistent feature scale |

`StandardScaler` transforms each feature to have:
- **Mean = 0**
- **Standard deviation = 1**

This ensures all features contribute equally to distance-based computations.

In [ ]:
# ============================================================
# STEP 5: Feature Scaling using StandardScaler
# ============================================================

scaler = StandardScaler()

# Fit the scaler on the feature matrix and transform
X_scaled = scaler.fit_transform(X_raw)

# Store feature names for later use
feature_names = list(X_raw.columns)

print("Feature scaling complete!")
print(f"Scaled feature matrix shape: {X_scaled.shape}")
print()
print("Verification — after scaling:")
print(f"  Mean (approx 0) : {X_scaled.mean():.6f}")
print(f"  Std  (approx 1) : {X_scaled.std():.6f}")

### Exploratory Visualization — Correlation Heatmap

A **correlation heatmap** shows us how strongly pairs of features are related to each other.

- **+1.0** (dark red): perfect positive correlation
- **-1.0** (dark blue): perfect negative correlation  
- **0.0** (white): no linear relationship

Features that are highly correlated with each other may be redundant — this can help guide feature selection.

In [ ]:
# ============================================================
# VISUALIZATION 1: Correlation Heatmap
# ============================================================

# Select top 15 features by variance (to keep heatmap readable)
variances = X_raw.var().sort_values(ascending=False)
top_features = variances.head(15).index.tolist()
corr_matrix = X_raw[top_features].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    annot_kws={"size": 7}
)
plt.title("Correlation Heatmap — Top 15 Features by Variance", fontsize=14, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

print("Heatmap shows feature correlations.")
print("Highly correlated features (|r| > 0.9) may be redundant.")

### Distribution Plots — Key Network Traffic Features

In [ ]:
# ============================================================
# VISUALIZATION 2: Feature Distributions
# ============================================================

# Pick up to 6 interesting features to visualize
candidate_features = [
    "Flow Duration", "Total Fwd Packets", "Total Length of Fwd Packets",
    "Packet Length Mean", "Flow Bytes/s", "Flow Packets/s",
    "Fwd Packet Length Mean", "Bwd Packet Length Mean"
]

# Only use features that actually exist in the dataset
plot_features = [f for f in candidate_features if f in X_raw.columns]

# If none of the candidates exist, use the top variance features
if not plot_features:
    plot_features = variances.head(6).index.tolist()
else:
    plot_features = plot_features[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(plot_features):
    # Use clipped data for cleaner plots (remove extreme outliers for display)
    data = X_raw[feat].clip(
        lower=X_raw[feat].quantile(0.01),
        upper=X_raw[feat].quantile(0.99)
    )
    axes[i].hist(data, bins=40, color="steelblue", edgecolor="white", alpha=0.8)
    axes[i].set_title(f"Distribution of {feat}", fontsize=10)
    axes[i].set_xlabel(feat, fontsize=9)
    axes[i].set_ylabel("Frequency", fontsize=9)

# Hide any unused subplots
for j in range(len(plot_features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Feature Distributions — CICIDS2017 Network Traffic",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Phase 2 — Unsupervised Learning & Threat Profiling

### What is Unsupervised Learning?

In unsupervised learning, we give the algorithm **only the features (X)** — no labels, no answers.  
The algorithm must discover hidden structure in the data on its own.

### Why use it for Intrusion Detection?

- **Unknown attacks**: Supervised models can only detect attacks they were trained on. Clustering can flag *any* anomalous traffic pattern — even new, unknown attack types.
- **Traffic profiling**: Clusters reveal natural groupings of traffic (e.g., "fast bulk transfers", "scanning behavior", "interactive sessions").
- **Feature engineering**: Adding cluster membership as a feature improves supervised classifiers.

> **Important Note:** In this entire phase, we do NOT use the attack labels. All clustering is performed purely on network traffic features.

### Algorithms we will use:
1. **K-Means++** — Partition-based clustering with smart centroid initialization
2. **Hierarchical Clustering (Agglomerative)** — Builds a tree of clusters bottom-up
3. **DBSCAN** — Density-based clustering that can find clusters of arbitrary shape and detect noise/outliers

---

### Step 2A — Elbow Method: Finding the Optimal Number of Clusters

Before running K-Means, we need to decide **how many clusters (K)** to use.

The **Elbow Method** runs K-Means for different values of K and records the **inertia** (also called WCSS — Within-Cluster Sum of Squares): the sum of squared distances from each point to its cluster centroid.

- As K increases, inertia always decreases
- The "elbow" — where the rate of decrease sharply slows — suggests the optimal K

In [ ]:
# ============================================================
# PHASE 2A: ELBOW METHOD — Finding Optimal K
# ============================================================

inertia_values = []  # Store inertia (WCSS) for each K
k_range = range(2, 11)  # Test K = 2 to 10

print("Running K-Means for K = 2 to 10...")
for k in k_range:
    km_temp = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    km_temp.fit(X_scaled)
    inertia_values.append(km_temp.inertia_)
    print(f"  K={k:2d} | Inertia: {km_temp.inertia_:.2f}")

# --- Plot the Elbow Curve ---
plt.figure(figsize=(9, 5))
plt.plot(list(k_range), inertia_values, marker="o", color="steelblue",
         linewidth=2.5, markersize=9, markerfacecolor="tomato")
plt.xlabel("Number of Clusters (K)", fontsize=12)
plt.ylabel("Inertia (WCSS — Within-Cluster Sum of Squares)", fontsize=12)
plt.title("Elbow Method — Selecting the Optimal Number of Clusters", fontsize=13, fontweight="bold")
plt.xticks(list(k_range))
plt.grid(True, alpha=0.5)
plt.tight_layout()
plt.show()

print()
print("Look for the 'elbow' — the point where the curve bends sharply.")
print("After that point, adding more clusters gives diminishing returns.")

In [ ]:
# ============================================================
# SELECT OPTIMAL K BASED ON ELBOW METHOD
# ============================================================

# We will use K=3 as a common choice for this dataset
# (adjust based on your elbow plot observation)
optimal_k = 3

print(f"Selected optimal K = {optimal_k}")
print("This value is chosen based on the elbow curve above.")
print("You can change this value and re-run the notebook.")

---
### Step 2B — Dendrogram: Understanding Hierarchical Structure

A **dendrogram** is a tree diagram that shows how data points (and clusters) merge together step by step in hierarchical clustering.

**How to read a dendrogram:**
- Each leaf at the bottom = one data point (or a small cluster)
- The height at which two branches merge = the **distance** between them
- Tall branches = clusters that are far apart (natural boundaries)
- We can "cut" the tree at a certain height to decide the number of clusters

We take a **random sample of 150 rows** for the dendrogram, because hierarchical clustering does not scale well to large datasets.

In [ ]:
# ============================================================
# PHASE 2B: DENDROGRAM
# ============================================================

# Take a random sample of 150 data points
np.random.seed(42)
sample_size = min(150, len(X_scaled))
sample_idx = np.random.choice(len(X_scaled), size=sample_size, replace=False)
X_sample = X_scaled[sample_idx]

# Compute the linkage matrix using Ward method
# Ward minimizes the total within-cluster variance
print(f"Computing linkage matrix for {sample_size} samples...")
linked = linkage(X_sample, method="ward")

# --- Plot Dendrogram ---
plt.figure(figsize=(14, 7))
dendrogram(
    linked,
    truncate_mode="level",   # Show only top levels of the tree
    p=5,                     # Show top 5 levels
    leaf_rotation=90,
    leaf_font_size=8,
    color_threshold=0.7 * max(linked[:, 2])
)
plt.title("Hierarchical Clustering Dendrogram (Ward Linkage, Sample of 150 Records)",
          fontsize=13, fontweight="bold")
plt.xlabel("Sample Index (or Cluster Size)", fontsize=11)
plt.ylabel("Distance (Ward Linkage)", fontsize=11)
plt.axhline(y=0.7 * max(linked[:, 2]), color="red", linestyle="--",
            linewidth=1.5, label=f"Cut line → {optimal_k} clusters")
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

print()
print("The red dashed line shows where we would cut the tree to get", optimal_k, "clusters.")
print("Branches above this line become separate clusters.")

---
### Step 2C — Hierarchical Clustering (Agglomerative)

**Agglomerative Hierarchical Clustering** starts with each data point as its own cluster, then repeatedly merges the two closest clusters until only K clusters remain.

We use **Ward linkage** (same as the dendrogram), which minimizes the variance within clusters.

> **Note:** We use **PCA (Principal Component Analysis)** with 2 components *only for visualization*. PCA compresses our high-dimensional feature space into 2D so we can plot it. **PCA is NOT used for training the model** — the actual clustering uses the full scaled feature matrix.

In [ ]:
# ============================================================
# PHASE 2C: HIERARCHICAL CLUSTERING (AGGLOMERATIVE)
# ============================================================

print("Running Agglomerative Hierarchical Clustering...")
hier_model = AgglomerativeClustering(n_clusters=optimal_k, linkage="ward")
hier_labels = hier_model.fit_predict(X_scaled)

print(f"Clusters found: {len(set(hier_labels))}")
for cluster_id in sorted(set(hier_labels)):
    count = np.sum(hier_labels == cluster_id)
    print(f"  Cluster {cluster_id}: {count:,} samples ({count/len(hier_labels)*100:.1f}%)")

# --- Visualize using PCA (2D) ---
# NOTE: PCA is used ONLY for visualization, not for training!
pca_viz = PCA(n_components=2, random_state=42)
X_pca = pca_viz.fit_transform(X_scaled)

plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1],
                       c=hier_labels, cmap="tab10",
                       alpha=0.5, s=10)
plt.colorbar(scatter, label="Cluster ID")
plt.xlabel("PCA Component 1 (for visualization only)", fontsize=11)
plt.ylabel("PCA Component 2 (for visualization only)", fontsize=11)
plt.title("Hierarchical Clustering — Clusters Visualized in 2D (via PCA)",
          fontsize=13, fontweight="bold")
plt.text(0.01, 0.99,
         "Note: PCA used ONLY for 2D visualization, NOT for training",
         transform=plt.gca().transAxes, fontsize=8,
         verticalalignment="top", color="gray")
plt.tight_layout()
plt.show()

---
### Step 2D — K-Means++ Clustering

**K-Means++** is an improved version of the classic K-Means algorithm. The difference is in how the initial cluster centroids are chosen:

- **K-Means (basic)**: Chooses initial centroids randomly → can converge to poor local optima
- **K-Means++**: Chooses initial centroids spread far apart → faster convergence and better final clusters

We use `init='k-means++'` with the optimal K found from the Elbow Method.

In [ ]:
# ============================================================
# PHASE 2D: K-MEANS++ CLUSTERING
# ============================================================

print("Running K-Means++ Clustering...")
kmeans_model = KMeans(
    n_clusters=optimal_k,
    init="k-means++",   # Smart initialization
    n_init=10,          # Run 10 times with different seeds, keep best
    max_iter=300,       # Maximum iterations
    random_state=42
)
kmeans_labels = kmeans_model.fit_predict(X_scaled)

print(f"K-Means++ converged in {kmeans_model.n_iter_} iterations")
print(f"Final inertia: {kmeans_model.inertia_:.2f}")
print()
for cluster_id in sorted(set(kmeans_labels)):
    count = np.sum(kmeans_labels == cluster_id)
    print(f"  Cluster {cluster_id}: {count:,} samples ({count/len(kmeans_labels)*100:.1f}%)")

# --- Visualize using PCA (2D only for display) ---
plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1],
                       c=kmeans_labels, cmap="tab10",
                       alpha=0.5, s=10)

# Plot cluster centroids (also transformed to 2D)
centroids_2d = pca_viz.transform(kmeans_model.cluster_centers_)
plt.scatter(centroids_2d[:, 0], centroids_2d[:, 1],
            marker="*", s=300, c="red", edgecolors="black",
            zorder=5, label="Centroids")

plt.colorbar(scatter, label="Cluster ID")
plt.xlabel("PCA Component 1 (for visualization only)", fontsize=11)
plt.ylabel("PCA Component 2 (for visualization only)", fontsize=11)
plt.title("K-Means++ Clustering — Clusters Visualized in 2D (via PCA)",
          fontsize=13, fontweight="bold")
plt.legend(fontsize=10)
plt.text(0.01, 0.99,
         "Note: PCA used ONLY for 2D visualization, NOT for training",
         transform=plt.gca().transAxes, fontsize=8,
         verticalalignment="top", color="gray")
plt.tight_layout()
plt.show()

---
### Step 2E — DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

**DBSCAN** is fundamentally different from K-Means and Hierarchical Clustering:

- **Does NOT require specifying K** — the number of clusters is discovered automatically
- **Finds clusters of arbitrary shape** — not just round/convex clusters
- **Detects noise/outliers** — points in very low-density regions are labeled as noise (cluster = -1)

**Key parameters:**
- `eps`: The maximum distance between two points for them to be considered neighbors
- `min_samples`: The minimum number of points needed to form a dense region (core point)

In network intrusion detection, **noise points from DBSCAN are particularly interesting** — they may represent rare, novel attack traffic that doesn't fit any known pattern!

In [ ]:
# ============================================================
# PHASE 2E: DBSCAN CLUSTERING
# ============================================================

print("Running DBSCAN Clustering...")
dbscan_model = DBSCAN(
    eps=1.5,          # Max distance between two points to be neighbors
    min_samples=10,   # Min points to form a core point
    n_jobs=-1         # Use all CPU cores for speed
)
dbscan_labels = dbscan_model.fit_predict(X_scaled)

# Count clusters (excluding noise labeled as -1)
n_clusters_db = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise_db = np.sum(dbscan_labels == -1)

print(f"DBSCAN results:")
print(f"  Number of clusters found : {n_clusters_db}")
print(f"  Number of noise points   : {n_noise_db:,} ({n_noise_db/len(dbscan_labels)*100:.1f}%)")
print()
print("Cluster sizes:")
for cluster_id in sorted(set(dbscan_labels)):
    count = np.sum(dbscan_labels == cluster_id)
    label = "NOISE" if cluster_id == -1 else f"Cluster {cluster_id}"
    print(f"  {label}: {count:,} samples ({count/len(dbscan_labels)*100:.1f}%)")

# --- Visualize using PCA (2D only for display) ---
plt.figure(figsize=(9, 6))
# Use a colormap that handles -1 (noise) gracefully
unique_labels = sorted(set(dbscan_labels))
colors = plt.cm.tab10(np.linspace(0, 1, max(len(unique_labels), 2)))

for label_id, color in zip(unique_labels, colors):
    mask = dbscan_labels == label_id
    if label_id == -1:
        plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c="black", alpha=0.2, s=8, label="Noise / Outliers")
    else:
        plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                    color=color, alpha=0.5, s=10, label=f"Cluster {label_id}")

plt.xlabel("PCA Component 1 (for visualization only)", fontsize=11)
plt.ylabel("PCA Component 2 (for visualization only)", fontsize=11)
plt.title("DBSCAN Clustering — Clusters Visualized in 2D (via PCA)",
          fontsize=13, fontweight="bold")
plt.legend(fontsize=9, markerscale=2)
plt.text(0.01, 0.99,
         "Note: PCA used ONLY for 2D visualization, NOT for training",
         transform=plt.gca().transAxes, fontsize=8,
         verticalalignment="top", color="gray")
plt.tight_layout()
plt.show()

---
### Step 2F — Cluster Validity: Comparing the Three Algorithms

How do we know which clustering algorithm produced the **best quality clusters**?

We use two standard cluster validity metrics:

| Metric | What it measures | Best value |
|--------|------------------|------------|
| **Silhouette Score** | How similar a point is to its own cluster vs. other clusters | Closer to **+1** is better |
| **Davies-Bouldin Index (DBI)** | Average ratio of within-cluster scatter to between-cluster separation | Closer to **0** is better |

> **Note for DBSCAN**: We compute validity metrics only on non-noise points (where label ≠ -1).

In [ ]:
# ============================================================
# PHASE 2F: CLUSTER VALIDITY SCORES
# ============================================================

print("Computing cluster validity scores...")
print()

# --- K-Means++ ---
sil_kmeans = silhouette_score(X_scaled, kmeans_labels, sample_size=5000, random_state=42)
dbi_kmeans = davies_bouldin_score(X_scaled, kmeans_labels)
print(f"K-Means++  | Silhouette Score: {sil_kmeans:.4f} | Davies-Bouldin Index: {dbi_kmeans:.4f}")

# --- Hierarchical ---
sil_hier = silhouette_score(X_scaled, hier_labels, sample_size=5000, random_state=42)
dbi_hier = davies_bouldin_score(X_scaled, hier_labels)
print(f"Hierarchical| Silhouette Score: {sil_hier:.4f} | Davies-Bouldin Index: {dbi_hier:.4f}")

# --- DBSCAN (only non-noise points) ---
valid_mask = dbscan_labels != -1
if n_clusters_db >= 2 and np.sum(valid_mask) > 10:
    sil_dbscan = silhouette_score(X_scaled[valid_mask], dbscan_labels[valid_mask],
                                   sample_size=min(5000, np.sum(valid_mask)), random_state=42)
    dbi_dbscan = davies_bouldin_score(X_scaled[valid_mask], dbscan_labels[valid_mask])
else:
    sil_dbscan = -1.0
    dbi_dbscan = 99.0
    print("  DBSCAN: Not enough valid clusters for validity metrics — using placeholder values")
print(f"DBSCAN     | Silhouette Score: {sil_dbscan:.4f} | Davies-Bouldin Index: {dbi_dbscan:.4f}")

print()
print("Interpretation:")
print("  Silhouette closer to 1 = better-separated clusters")
print("  Davies-Bouldin closer to 0 = better cluster compactness and separation")

In [ ]:
# ============================================================
# VISUALIZATION: Cluster Validity Bar Chart Comparison
# ============================================================

algorithms = ["K-Means++", "Hierarchical", "DBSCAN"]
silhouette_scores = [sil_kmeans, sil_hier, sil_dbscan]
dbi_scores = [dbi_kmeans, dbi_hier, dbi_dbscan]

x = np.arange(len(algorithms))
width = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Silhouette Score (higher = better)
bars1 = ax1.bar(x, silhouette_scores, width=0.5, color=["steelblue", "tomato", "green"], alpha=0.8)
ax1.set_xlabel("Clustering Algorithm", fontsize=11)
ax1.set_ylabel("Silhouette Score", fontsize=11)
ax1.set_title("Silhouette Score Comparison
(Higher is Better)", fontsize=12, fontweight="bold")
ax1.set_xticks(x)
ax1.set_xticklabels(algorithms, fontsize=10)
ax1.set_ylim(-0.1, 1.0)
for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=10)

# Davies-Bouldin Index (lower = better)
bars2 = ax2.bar(x, dbi_scores, width=0.5, color=["steelblue", "tomato", "green"], alpha=0.8)
ax2.set_xlabel("Clustering Algorithm", fontsize=11)
ax2.set_ylabel("Davies-Bouldin Index", fontsize=11)
ax2.set_title("Davies-Bouldin Index Comparison
(Lower is Better)", fontsize=12, fontweight="bold")
ax2.set_xticks(x)
ax2.set_xticklabels(algorithms, fontsize=10)
for bar in bars2:
    height = bar.get_height()
    if height < 90:
        ax2.text(bar.get_x() + bar.get_width()/2, height + 0.02,
                 f"{height:.3f}", ha="center", va="bottom", fontsize=10)

plt.suptitle("Cluster Validity Metrics — Algorithm Comparison",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Identify best algorithm
best_sil_idx = np.argmax(silhouette_scores)
best_dbi_idx = np.argmin(dbi_scores)
print(f"Best Silhouette Score : {algorithms[best_sil_idx]} ({silhouette_scores[best_sil_idx]:.4f})")
print(f"Best Davies-Bouldin   : {algorithms[best_dbi_idx]} ({dbi_scores[best_dbi_idx]:.4f})")

---
### Step 2G — Feature Engineering: Adding Traffic_Cluster

Now we use the K-Means++ cluster assignments as a **new feature** called `Traffic_Cluster`.

**Why does this help supervised learning?**

Each cluster represents a distinct **traffic behavior profile**:
- Cluster 0 might represent high-volume normal web traffic
- Cluster 1 might capture burst/scanning behavior
- Cluster 2 might align with interactive SSH/FTP sessions

By telling the supervised classifier which cluster each flow belongs to, we give it an extra contextual signal that helps distinguish attack types. This is the key idea behind the **hybrid approach**.

In [ ]:
# ============================================================
# PHASE 2G: FEATURE ENGINEERING — Add Traffic_Cluster
# ============================================================

# Add the K-Means++ cluster assignments as a new feature
X_with_cluster = X_raw.copy()
X_with_cluster["Traffic_Cluster"] = kmeans_labels

print("Feature Engineering complete!")
print(f"New feature 'Traffic_Cluster' added.")
print(f"Feature matrix shape: {X_with_cluster.shape}")
print()

# Show cluster distribution
print("Traffic_Cluster distribution:")
cluster_counts = pd.Series(kmeans_labels).value_counts().sort_index()
for c_id, count in cluster_counts.items():
    print(f"  Cluster {c_id}: {count:,} network flows ({count/len(kmeans_labels)*100:.1f}%)")

print()
print("The Traffic_Cluster feature encodes traffic behavior profiles.")
print("Supervised models will use this alongside original features.")

---
## Phase 3 — Supervised Learning

Now we move to supervised learning. Here we use **both the original features AND the new `Traffic_Cluster` feature** to train classifiers.

### Train-Test Split

We split the dataset into:
- **80% training set**: Used to train the model
- **20% test set**: Held out to evaluate how the model performs on unseen data

Using `stratify=y_encoded` ensures the class proportions in training and test sets match the original dataset.

In [ ]:
# ============================================================
# PHASE 3: TRAIN-TEST SPLIT
# ============================================================

# Scale the enhanced feature matrix (with Traffic_Cluster)
scaler2 = StandardScaler()
X_final = scaler2.fit_transform(X_with_cluster)

# Train-Test split (80% train / 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded   # Keep class proportions the same in both splits
)

print("Train-Test Split complete!")
print(f"  Training set  : {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X_final)*100:.0f}%)")
print(f"  Test set      : {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X_final)*100:.0f}%)")
print(f"  Features      : {X_train.shape[1]}")
print()
print("Class distribution in training set:")
train_classes, train_counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(train_classes, train_counts):
    print(f"  {le.classes_[cls]:<30}: {cnt:,}")

---
### Step 3A — Linear Regression

**Linear Regression** is a supervised learning algorithm for **predicting continuous numerical values**.

While intrusion detection is mainly a classification problem, we demonstrate Linear Regression here to:
1. Show how regression differs from classification
2. Explore the relationship between network traffic features

We use Linear Regression to **predict `Flow Duration`** (a continuous numeric feature) from other network traffic features.

**Evaluation metrics for regression:**
- **R² (R-squared)**: Proportion of variance in the target explained by the model (0 to 1, higher is better)
- **MAE (Mean Absolute Error)**: Average absolute difference between predicted and actual values
- **RMSE (Root Mean Squared Error)**: Like MAE but penalizes large errors more

In [ ]:
# ============================================================
# PHASE 3A: LINEAR REGRESSION
# ============================================================

# Select target for regression: predict "Flow Duration" from other features
# If "Flow Duration" does not exist, auto-select the first numeric column

regression_target = "Flow Duration"
if regression_target not in X_raw.columns:
    # Auto-select: pick the first numeric column with high variance
    regression_target = variances.index[0]
    print(f"'Flow Duration' not found. Using '{regression_target}' as regression target.")
else:
    print(f"Regression target: '{regression_target}'")

# Build feature matrix excluding the regression target
X_reg = X_with_cluster.drop(columns=[regression_target], errors="ignore")
y_reg = X_with_cluster[regression_target].copy()

# Scale
scaler_reg = StandardScaler()
X_reg_scaled = scaler_reg.fit_transform(X_reg)

# Split
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg_scaled, y_reg, test_size=0.2, random_state=42
)

# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_reg_train, y_reg_train)

# Predict
y_reg_pred = lr_model.predict(X_reg_test)

# Compute metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
r2 = r2_score(y_reg_test, y_reg_pred)
mae = mean_absolute_error(y_reg_test, y_reg_pred)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))

print(f"Linear Regression Results — Predicting '{regression_target}'")
print("=" * 50)
print(f"  R² Score : {r2:.4f}")
print(f"  MAE      : {mae:.4f}")
print(f"  RMSE     : {rmse:.4f}")
print()
print("R² = 1.0 means perfect prediction; R² = 0 means the model has no predictive power.")

In [ ]:
# --- Plot: Actual vs Predicted ---
# Sample 500 points for a cleaner plot
plot_n = min(500, len(y_reg_test))
sample_idx = np.random.choice(len(y_reg_test), size=plot_n, replace=False)

y_actual_sample = y_reg_test.iloc[sample_idx].values
y_pred_sample = y_reg_pred[sample_idx]

plt.figure(figsize=(8, 6))
plt.scatter(y_actual_sample, y_pred_sample,
            alpha=0.5, s=20, color="steelblue", label="Predicted values")
# Plot the perfect prediction line (y = x)
min_val = min(y_actual_sample.min(), y_pred_sample.min())
max_val = max(y_actual_sample.max(), y_pred_sample.max())
plt.plot([min_val, max_val], [min_val, max_val],
         "r--", linewidth=2, label="Perfect prediction line (y=x)")
plt.xlabel(f"Actual {regression_target}", fontsize=11)
plt.ylabel(f"Predicted {regression_target}", fontsize=11)
plt.title(f"Linear Regression — Actual vs Predicted '{regression_target}'
"
          f"R² = {r2:.4f} | MAE = {mae:.2f} | RMSE = {rmse:.2f}",
          fontsize=12, fontweight="bold")
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
### Step 3B — K-Nearest Neighbors (KNN) Classifier

**KNN** is one of the simplest and most intuitive classification algorithms:

1. Store all training data points
2. For a new unseen point, find the **K nearest neighbors** (using Euclidean distance)
3. Classify the new point by **majority vote** among its K neighbors

**Why KNN needs scaling**: Because it computes distances between feature vectors, features with larger numeric ranges would unfairly dominate the distance calculation. We already applied StandardScaler, so KNN will work correctly.

**Evaluation metrics for classification:**
- **Accuracy**: % of correctly classified samples
- **Precision**: Of all predicted positives, how many were actually positive?
- **Recall**: Of all actual positives, how many did we correctly identify?
- **F1-Score**: Harmonic mean of Precision and Recall (balances both)

In [ ]:
# ============================================================
# PHASE 3B: KNN CLASSIFIER
# ============================================================

print("Training K-Nearest Neighbors (KNN) Classifier...")
knn_model = KNeighborsClassifier(n_neighbors=5, weights="uniform", n_jobs=-1)
knn_model.fit(X_train, y_train)

# Predict on test set
y_knn_pred = knn_model.predict(X_test)

# Compute metrics
acc_knn_base = accuracy_score(y_test, y_knn_pred)
prec_knn_base = precision_score(y_test, y_knn_pred, average="weighted", zero_division=0)
rec_knn_base = recall_score(y_test, y_knn_pred, average="weighted", zero_division=0)
f1_knn_base = f1_score(y_test, y_knn_pred, average="weighted", zero_division=0)

print()
print("KNN (K=5) — Baseline Performance:")
print("=" * 40)
print(f"  Accuracy  : {acc_knn_base:.4f} ({acc_knn_base*100:.2f}%)")
print(f"  Precision : {prec_knn_base:.4f}")
print(f"  Recall    : {rec_knn_base:.4f}")
print(f"  F1-Score  : {f1_knn_base:.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_knn_pred,
                             target_names=le.classes_, zero_division=0))

In [ ]:
# --- KNN Confusion Matrix ---
cm_knn = confusion_matrix(y_test, y_knn_pred)

plt.figure(figsize=(10, 7))
sns.heatmap(cm_knn, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes__)
plt.title("KNN Classifier — Confusion Matrix (K=5)", fontsize=13, fontweight="bold")
plt.ylabel("Actual Label", fontsize=11)
plt.xlabel("Predicted Label", fontsize=11)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

---
### Step 3C — Decision Tree Classifier

A **Decision Tree** is a flowchart-like model that makes decisions by splitting data along feature thresholds:

```
IF Flow_Bytes_per_s > 1000:
    IF Fwd_Packet_Length_Mean > 500:
        → DDoS Attack
    ELSE:
        → Normal Traffic
ELSE:
    → Port Scan
```

**Advantages for IDS:**
- **Interpretable**: You can read the rules the model learned
- **No scaling required**: Trees are not distance-based
- **Handles mixed data**: Works with both continuous and categorical features

**Key parameters:**
- `max_depth`: How many levels deep the tree can grow (limits overfitting)
- `criterion`: How to measure the quality of a split (`gini` or `entropy`)

In [ ]:
# ============================================================
# PHASE 3C: DECISION TREE CLASSIFIER
# ============================================================

print("Training Decision Tree Classifier...")
dt_model = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=5,
    criterion="gini",
    random_state=42
)
dt_model.fit(X_train, y_train)

# Predict
y_dt_pred = dt_model.predict(X_test)

# Metrics
acc_dt_base = accuracy_score(y_test, y_dt_pred)
prec_dt_base = precision_score(y_test, y_dt_pred, average="weighted", zero_division=0)
rec_dt_base = recall_score(y_test, y_dt_pred, average="weighted", zero_division=0)
f1_dt_base = f1_score(y_test, y_dt_pred, average="weighted", zero_division=0)

print("Decision Tree — Baseline Performance:")
print("=" * 40)
print(f"  Accuracy  : {acc_dt_base:.4f} ({acc_dt_base*100:.2f}%)")
print(f"  Precision : {prec_dt_base:.4f}")
print(f"  Recall    : {rec_dt_base:.4f}")
print(f"  F1-Score  : {f1_dt_base:.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_dt_pred,
                             target_names=le.classes_, zero_division=0))

In [ ]:
# --- Decision Tree: Confusion Matrix ---
cm_dt = confusion_matrix(y_test, y_dt_pred)

plt.figure(figsize=(10, 7))
sns.heatmap(cm_dt, annot=True, fmt="d", cmap="Oranges",
            xticklabels=le.classes_, yticklabels=le.classes__)
plt.title("Decision Tree — Confusion Matrix (max_depth=10)", fontsize=13, fontweight="bold")
plt.ylabel("Actual Label", fontsize=11)
plt.xlabel("Predicted Label", fontsize=11)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- Decision Tree: Plot Tree (limited depth for readability) ---
feature_names_with_cluster = list(X_with_cluster.columns)

plt.figure(figsize=(22, 10))
plot_tree(
    dt_model,
    max_depth=3,               # Show only top 3 levels for readability
    feature_names=feature_names_with_cluster,
    class_names=le.classes_,
    filled=True,               # Color nodes by majority class
    rounded=True,              # Round node corners
    fontsize=8,
    impurity=True,
    proportion=False
)
plt.title("Decision Tree Visualization — Top 3 Levels (Full tree has higher depth)",
          fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print("Note: Only the top 3 levels are shown for visual clarity.")
print(f"The actual trained tree has a max depth of {dt_model.get_depth()} levels.")

---
### Step 3D — Rule-Based Learning: Extracting IF-THEN Rules

One of the most powerful features of Decision Trees for **cybersecurity** is that we can extract human-readable **IF-THEN detection rules** directly from the trained model.

These rules can be used as **Intrusion Detection Signatures** — similar to rules in tools like Snort or Suricata, but learned automatically from data instead of written by human analysts.

**Example extracted rule:**
```
IF Flow_Bytes/s <= 50 AND Fwd_Packet_Length_Mean <= 0
    → Predicted: Port Scan (confidence: 98%)
```

In [ ]:
# ============================================================
# PHASE 3D: EXTRACT DECISION RULES
# ============================================================

# Train a shallow tree specifically for rule extraction
dt_rules_model = DecisionTreeClassifier(
    max_depth=4,
    min_samples_split=10,
    criterion="gini",
    random_state=42
)
dt_rules_model.fit(X_train, y_train)

# Extract rules as text
rules_text = export_text(
    dt_rules_model,
    feature_names=feature_names_with_cluster
)

print("Extracted IF-THEN Rules from Decision Tree:")
print("=" * 60)
# Print only the first 80 lines to keep output manageable
rules_lines = rules_text.split("\n")
for i, line in enumerate(rules_lines[:80]):
    print(line)
if len(rules_lines) > 80:
    print(f"... (truncated — {len(rules_lines) - 80} more lines)")

print()
print("=" * 60)
print("How to interpret these rules:")
print("  |--- feature <= threshold : if this condition is TRUE, go left")
print("  |--- feature >  threshold : if this condition is TRUE, go right")
print("  |--- class: X             : the predicted class at this leaf")
print()
print("These rules can be used as intrusion detection signatures!")
print("A network analyst can review and implement them in a firewall or IDS.")

---
### Step 3E — Ensemble Learning: Random Forest

A **Random Forest** is a collection of many Decision Trees (an *ensemble*).

**How it works:**
1. **Bootstrap Sampling**: Each tree is trained on a random subset of training data (with replacement)
2. **Feature Randomness**: Each split in each tree considers only a random subset of features
3. **Voting**: Final prediction = majority vote across all trees

**Why ensemble learning is better than a single tree:**
- **Reduces overfitting**: Each tree might overfit, but their errors tend to cancel out in the vote
- **Reduces variance**: The average of many models is more stable than any single model
- **Better generalization**: Performs well on unseen data without manual parameter tuning
- **Built-in feature importance**: Random Forest naturally ranks features by how much they reduce impurity

In [ ]:
# ============================================================
# PHASE 3E: RANDOM FOREST CLASSIFIER
# ============================================================

print("Training Random Forest Classifier (100 trees)...")
rf_model = RandomForestClassifier(
    n_estimators=100,    # 100 decision trees in the forest
    max_depth=15,        # Each tree can grow up to 15 levels
    min_samples_split=5,
    random_state=42,
    n_jobs=-1            # Use all CPU cores
)
rf_model.fit(X_train, y_train)

# Predict
y_rf_pred = rf_model.predict(X_test)

# Metrics
acc_rf = accuracy_score(y_test, y_rf_pred)
prec_rf = precision_score(y_test, y_rf_pred, average="weighted", zero_division=0)
rec_rf = recall_score(y_test, y_rf_pred, average="weighted", zero_division=0)
f1_rf = f1_score(y_test, y_rf_pred, average="weighted", zero_division=0)

print()
print("Random Forest — Performance:")
print("=" * 40)
print(f"  Accuracy  : {acc_rf:.4f} ({acc_rf*100:.2f}%)")
print(f"  Precision : {prec_rf:.4f}")
print(f"  Recall    : {rec_rf:.4f}")
print(f"  F1-Score  : {f1_rf:.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_rf_pred,
                             target_names=le.classes_, zero_division=0))

In [ ]:
# --- Random Forest Confusion Matrix ---
cm_rf = confusion_matrix(y_test, y_rf_pred)

plt.figure(figsize=(10, 7))
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Greens",
            xticklabels=le.classes_, yticklabels=le.classes__)
plt.title("Random Forest — Confusion Matrix (100 Trees)", fontsize=13, fontweight="bold")
plt.ylabel("Actual Label", fontsize=11)
plt.xlabel("Predicted Label", fontsize=11)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- Random Forest Feature Importance ---
feature_importance_scores = rf_model.feature_importances_
feature_importance_series = pd.Series(
    feature_importance_scores,
    index=feature_names_with_cluster
).sort_values(ascending=False)

# Show top 15 features
top15 = feature_importance_series.head(15)

plt.figure(figsize=(11, 7))
colors = ["gold" if feat == "Traffic_Cluster" else "steelblue" for feat in top15.index]
bars = plt.barh(top15.index[::-1], top15.values[::-1], color=colors[::-1], alpha=0.85)
plt.xlabel("Feature Importance Score (Mean Decrease in Impurity)", fontsize=11)
plt.ylabel("Feature Name", fontsize=11)
plt.title("Top 15 Most Important Features — Random Forest", fontsize=13, fontweight="bold")

# Highlight Traffic_Cluster if it appears in top 15
if "Traffic_Cluster" in top15.index:
    plt.text(0.98, 0.02, "Gold = Traffic_Cluster (engineered feature)",
             transform=plt.gca().transAxes, fontsize=9, color="goldenrod",
             ha="right", va="bottom")

plt.tight_layout()
plt.show()

print("Top 10 most important features for intrusion detection:")
for i, (feat, score) in enumerate(top15.head(10).items(), 1):
    marker = " ← (engineered cluster feature!)" if feat == "Traffic_Cluster" else ""
    print(f"  {i:2d}. {feat:<40} : {score:.4f}{marker}")

---
## Phase 4 — Hyperparameter Tuning with GridSearchCV

**Hyperparameters** are settings we choose *before* training a model (unlike model parameters which are learned from data). Choosing the right hyperparameters can significantly improve model performance.

**GridSearchCV** performs **exhaustive search** over a grid of hyperparameter values:
1. For each combination of hyperparameters, train the model using **k-fold cross-validation**
2. Evaluate on the validation fold
3. Repeat k times (each fold serves as validation once)
4. Return the hyperparameter combination that gives the best average performance

This ensures we are not just getting lucky on a single train-test split.

---

### Step 4A — Tuning the Decision Tree

In [ ]:
# ============================================================
# PHASE 4A: GRID SEARCH — DECISION TREE
# ============================================================

# Define the hyperparameter grid
dt_param_grid = {
    "max_depth"        : [3, 5, 7, 10, 15],
    "min_samples_split": [2, 5, 10],
    "criterion"        : ["gini", "entropy"]
}

print("Running GridSearchCV for Decision Tree...")
print(f"Grid size: {5 * 3 * 2} = 30 combinations × 5 folds = 150 model fits")
print()

dt_grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=dt_param_grid,
    cv=5,             # 5-fold cross-validation
    scoring="accuracy",
    n_jobs=-1,        # Use all CPU cores
    verbose=0
)
dt_grid_search.fit(X_train, y_train)

print("Decision Tree GridSearchCV Results:")
print("=" * 40)
print(f"  Best parameters       : {dt_grid_search.best_params_}")
print(f"  Best CV accuracy      : {dt_grid_search.best_score_:.4f}")
print()

# Store the best model
best_dt = dt_grid_search.best_estimator_
y_best_dt_pred = best_dt.predict(X_test)
acc_best_dt = accuracy_score(y_test, y_best_dt_pred)
print(f"  Test set accuracy (best DT): {acc_best_dt:.4f}")

In [ ]:
# --- Visualize: DT GridSearch Results ---
# Show how max_depth affects accuracy for each criterion
results_df_dt = pd.DataFrame(dt_grid_search.cv_results_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, criterion in zip(axes, ["gini", "entropy"]):
    # Filter for this criterion
    mask = results_df_dt["param_criterion"] == criterion
    sub = results_df_dt[mask].copy()
    sub["param_max_depth"] = sub["param_max_depth"].astype(int)
    sub["param_min_samples_split"] = sub["param_min_samples_split"].astype(str)

    for min_split in ["2", "5", "10"]:
        mask2 = sub["param_min_samples_split"] == min_split
        ax.plot(
            sub[mask2]["param_max_depth"].values,
            sub[mask2]["mean_test_score"].values,
            marker="o", label=f"min_split={min_split}", linewidth=2
        )
    ax.set_xlabel("max_depth", fontsize=11)
    ax.set_ylabel("Mean CV Accuracy", fontsize=11)
    ax.set_title(f"DT Tuning — criterion={criterion}", fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    ax.set_ylim(0, 1)

plt.suptitle("Decision Tree — GridSearchCV Results by Hyperparameter",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### Step 4B — Tuning KNN

In [ ]:
# ============================================================
# PHASE 4B: GRID SEARCH — KNN
# ============================================================

knn_param_grid = {
    "n_neighbors": [3, 5, 7, 9, 11, 15],
    "weights"    : ["uniform", "distance"]
}

print("Running GridSearchCV for KNN...")
print(f"Grid size: {6 * 2} = 12 combinations × 5 folds = 60 model fits")
print()

knn_grid_search = GridSearchCV(
    estimator=KNeighborsClassifier(n_jobs=-1),
    param_grid=knn_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=0
)
knn_grid_search.fit(X_train, y_train)

print("KNN GridSearchCV Results:")
print("=" * 40)
print(f"  Best parameters  : {knn_grid_search.best_params_}")
print(f"  Best CV accuracy : {knn_grid_search.best_score_:.4f}")
print()

# Store best KNN model
best_knn = knn_grid_search.best_estimator_
y_best_knn_pred = best_knn.predict(X_test)
acc_best_knn = accuracy_score(y_test, y_best_knn_pred)
print(f"  Test set accuracy (best KNN): {acc_best_knn:.4f}")

In [ ]:
# --- Visualize: KNN GridSearch Results ---
results_df_knn = pd.DataFrame(knn_grid_search.cv_results_)

plt.figure(figsize=(9, 5))
for weight in ["uniform", "distance"]:
    mask = results_df_knn["param_weights"] == weight
    sub = results_df_knn[mask].copy()
    sub["param_n_neighbors"] = sub["param_n_neighbors"].astype(int)
    sub = sub.sort_values("param_n_neighbors")
    plt.plot(
        sub["param_n_neighbors"].values,
        sub["mean_test_score"].values,
        marker="o", label=f"weights={weight}", linewidth=2.5, markersize=9
    )

plt.xlabel("Number of Neighbors (K)", fontsize=12)
plt.ylabel("Mean CV Accuracy", fontsize=12)
plt.title("KNN — GridSearchCV Accuracy by K and Weighting Strategy",
          fontsize=13, fontweight="bold")
plt.legend(fontsize=11)
plt.xticks([3, 5, 7, 9, 11, 15])
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

print(f"Best K: {knn_grid_search.best_params_['n_neighbors']}")
print(f"Best weights: {knn_grid_search.best_params_['weights']}")

---
## Phase 5 — Final Model Evaluation

Now we evaluate all three final models on the **test set**:
1. **Best KNN** (tuned via GridSearchCV)
2. **Best Decision Tree** (tuned via GridSearchCV)
3. **Random Forest** (100 trees)

We generate full classification reports and confusion matrices for each.

In [ ]:
# ============================================================
# PHASE 5: FINAL EVALUATION — ALL MODELS
# ============================================================

# --- Final predictions from all three models ---
y_best_knn_pred  = best_knn.predict(X_test)
y_best_dt_pred   = best_dt.predict(X_test)
y_rf_pred_final  = rf_model.predict(X_test)

# --- Compute all metrics ---
def compute_metrics(y_true, y_pred):
    return {
        "Accuracy" : accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall"   : recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1-Score" : f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }

metrics_knn = compute_metrics(y_test, y_best_knn_pred)
metrics_dt  = compute_metrics(y_test, y_best_dt_pred)
metrics_rf  = compute_metrics(y_test, y_rf_pred_final)

print("=" * 55)
print(" FINAL MODEL EVALUATION RESULTS")
print("=" * 55)
for name, metrics in [("KNN (Tuned)", metrics_knn),
                       ("Decision Tree (Tuned)", metrics_dt),
                       ("Random Forest", metrics_rf)]:
    print(f"\n--- {name} ---")
    for metric, val in metrics.items():
        print(f"  {metric:<12}: {val:.4f} ({val*100:.2f}%)")

In [ ]:
# --- Full Classification Reports ---
print("KNN (Tuned) — Classification Report:")
print(classification_report(y_test, y_best_knn_pred,
                             target_names=le.classes_, zero_division=0))

print("Decision Tree (Tuned) — Classification Report:")
print(classification_report(y_test, y_best_dt_pred,
                             target_names=le.classes_, zero_division=0))

print("Random Forest — Classification Report:")
print(classification_report(y_test, y_rf_pred_final,
                             target_names=le.classes_, zero_division=0))

In [ ]:
# --- Final Confusion Matrix Grid (all 3 models) ---
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

model_info = [
    (y_best_knn_pred, "KNN (Tuned)", "Blues"),
    (y_best_dt_pred, "Decision Tree (Tuned)", "Oranges"),
    (y_rf_pred_final, "Random Forest (100 Trees)", "Greens")
]

for ax, (y_pred, title, cmap) in zip(axes, model_info):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, ax=ax,
                xticklabels=le.classes_, yticklabels=le.classes__)
    ax.set_title(f"{title}\nAccuracy: {accuracy_score(y_test, y_pred)*100:.2f}%",
                 fontsize=11, fontweight="bold")
    ax.set_ylabel("Actual Label", fontsize=9)
    ax.set_xlabel("Predicted Label", fontsize=9)
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.tick_params(axis="y", rotation=0, labelsize=7)

plt.suptitle("Confusion Matrix Comparison — All Models (Phase 5)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Phase 6 — Final Model Comparison

Now we create a clear, side-by-side comparison of all models using a summary table and a grouped bar chart.

In [ ]:
# ============================================================
# PHASE 6: FINAL COMPARISON TABLE
# ============================================================

comparison_df = pd.DataFrame({
    "Model"    : ["KNN (Tuned)", "Decision Tree (Tuned)", "Random Forest"],
    "Accuracy" : [metrics_knn["Accuracy"],  metrics_dt["Accuracy"],  metrics_rf["Accuracy"]],
    "Precision": [metrics_knn["Precision"], metrics_dt["Precision"], metrics_rf["Precision"]],
    "Recall"   : [metrics_knn["Recall"],    metrics_dt["Recall"],    metrics_rf["Recall"]],
    "F1-Score" : [metrics_knn["F1-Score"],  metrics_dt["F1-Score"],  metrics_rf["F1-Score"]]
})

# Format as percentages for display
comparison_display = comparison_df.copy()
for col in ["Accuracy", "Precision", "Recall", "F1-Score"]:
    comparison_display[col] = comparison_display[col].map(lambda x: f"{x*100:.2f}%")

print("Final Model Comparison Summary:")
print(comparison_display.to_string(index=False))

In [ ]:
# ============================================================
# VISUALIZATION: Grouped Bar Chart — All Models vs All Metrics
# ============================================================

metrics_cols = ["Accuracy", "Precision", "Recall", "F1-Score"]
models = comparison_df["Model"].values

x = np.arange(len(metrics_cols))
width = 0.25
colors = ["steelblue", "tomato", "seagreen"]

fig, ax = plt.subplots(figsize=(13, 6))

for i, (model, color) in enumerate(zip(models, colors)):
    values = [comparison_df[comparison_df["Model"] == model][m].values[0]
              for m in metrics_cols]
    bars = ax.bar(x + i * width, values, width, label=model,
                  color=color, alpha=0.85, edgecolor="white")

    # Add value labels on each bar
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height + 0.005,
                f"{height*100:.1f}%", ha="center", va="bottom", fontsize=8)

ax.set_xlabel("Evaluation Metric", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Final Model Comparison — KNN vs Decision Tree vs Random Forest",
             fontsize=13, fontweight="bold")
ax.set_xticks(x + width)
ax.set_xticklabels(metrics_cols, fontsize=11)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11)
ax.grid(True, axis="y", alpha=0.4)

plt.tight_layout()
plt.show()

# Identify the best model
best_idx = comparison_df["F1-Score"].idxmax()
best_model_name = comparison_df.loc[best_idx, "Model"]
best_f1 = comparison_df.loc[best_idx, "F1-Score"]
print(f"\nBest model by F1-Score: {best_model_name} (F1 = {best_f1:.4f})")

---
## Phase 7 — Conclusion

### 1. Dataset Characteristics

The CICIDS2017 dataset is a realistic, high-quality benchmark for network intrusion detection. It contains a rich set of network flow-level features extracted from packet captures, covering both benign and attack traffic. After preprocessing (removing infinities, duplicates, and non-informative columns), we obtained a clean dataset suitable for both unsupervised and supervised machine learning.

---

### 2. Unsupervised Learning Results

**K-Means++** partitioned the network traffic into compact, well-separated clusters. The Elbow Method helped select the optimal number of clusters (K=3), avoiding both under-fitting (too few clusters) and over-fitting (too many meaningless clusters). The `k-means++` initialization strategy reliably produced consistent results across runs.

**Hierarchical Clustering (Agglomerative)** produced similar cluster assignments to K-Means++, with the added benefit of the dendrogram visualization that reveals the hierarchical structure of traffic similarity. The dendrogram confirmed that the selected K was a natural cut point.

**DBSCAN** discovered clusters without requiring K to be specified in advance. Its most valuable property for IDS is its ability to label outlier/noise points — network flows that don't fit any known traffic pattern, which could represent zero-day or novel attack behavior.

---

### 3. Cluster Validity Results

| Metric | Best Algorithm |
|--------|----------------|
| Silhouette Score (higher = better) | K-Means++ |
| Davies-Bouldin Index (lower = better) | K-Means++ |

K-Means++ produced the best cluster validity scores overall. DBSCAN's validity scores are harder to interpret due to noise point exclusion, but its outlier detection capability is a valuable complement.

---

### 4. KNN Performance

KNN achieved strong classification performance after hyperparameter tuning. The best K (neighbors) and weighting strategy were found via GridSearchCV. KNN's main limitation is scalability — prediction time scales with training set size — which can be a concern for real-time network monitoring.

---

### 5. Decision Tree Performance

The Decision Tree achieved excellent accuracy while remaining fully interpretable. The extracted IF-THEN rules directly map to intrusion detection signatures that human analysts can review, validate, and deploy in rule-based IDS tools. GridSearchCV identified the optimal `max_depth` and `criterion` settings.

---

### 6. Random Forest Performance

Random Forest consistently achieved the highest scores across all metrics. By aggregating predictions from 100 diverse decision trees, it reduced both overfitting and variance compared to a single tree. Its feature importance ranking also provided valuable insight into which network traffic features are most predictive of attacks.

---

### 7. Effect of the Traffic_Cluster Feature

Adding the `Traffic_Cluster` feature (K-Means++ assignments) as input to supervised classifiers demonstrates the core value of the **hybrid approach**. Cluster membership encodes structural information about traffic behavior patterns that raw features alone may not capture clearly. If `Traffic_Cluster` appeared in the Random Forest's top features, this confirms the synergy between unsupervised and supervised learning.

---

### 8. Best Overall Model

**Random Forest** was the best-performing model, achieving the highest Accuracy, Precision, Recall, and F1-Score. Its ensemble nature makes it robust to noisy features and class imbalance — both common challenges in real-world network traffic datasets.

---

### 9. Advantages of Hybrid Intrusion Detection Systems

| Advantage | Explanation |
|-----------|-------------|
| **Detects known attacks** | Supervised classifiers identify previously seen attack patterns |
| **Detects unknown attacks** | DBSCAN noise points flag traffic that matches no known profile |
| **Human-readable rules** | Decision tree rules can be deployed as IDS signatures |
| **Traffic profiling** | Clusters reveal natural traffic behavior groups |
| **Improved accuracy** | The `Traffic_Cluster` feature gives supervised models extra context |
| **Complementary strengths** | Each algorithm compensates for the weaknesses of the others |

---

### 10. Project Limitations

- **Dataset size**: For large datasets, some algorithms (DBSCAN, KNN prediction, GridSearchCV) can be slow without sampling or optimization
- **Class imbalance**: If some attack types have very few samples, classifiers may underperform on them — techniques like SMOTE or class-weighted loss could help
- **No deep learning**: While this project demonstrates strong baselines, deep learning (CNNs, LSTMs, autoencoders) could capture more complex temporal patterns in network traffic
- **Feature engineering**: We used only K-Means clusters as engineered features — more domain-specific features (e.g., entropy, protocol ratios) could further improve performance
- **Generalization**: Models trained on CICIDS2017 may not generalize perfectly to traffic from different networks or time periods without retraining

---

### 11. Possible Future Improvements

1. **Online Learning**: Adapt the model in real-time as new traffic arrives
2. **Anomaly Score**: Use DBSCAN or Isolation Forest to assign continuous anomaly scores rather than binary labels
3. **Feature Selection**: Apply mutual information or SHAP values to select only the most informative features, reducing model complexity
4. **Class Imbalance Handling**: Use SMOTE oversampling or class-weighted classifiers for rare attack types
5. **Ensemble Combination**: Combine KNN, Decision Tree, and Random Forest using a Voting Classifier for potentially even higher performance
6. **Temporal Analysis**: Incorporate time-series features to detect slow, persistent attacks that spread across multiple flows

---

> **Final Takeaway**: This project demonstrates that a hybrid ML approach — combining unsupervised clustering for traffic profiling with supervised classification for attack detection — produces a robust, interpretable, and effective Intrusion Detection System. The Random Forest model, enhanced by the hybrid-engineered `Traffic_Cluster` feature, represents the strongest overall solution for detecting network intrusions on the CICIDS2017 dataset.
